# DiT架构源码解析

DiT的核心架构如下图所示：

<div align="center">
    <img src="./imgs/DiT_architecture.png" alt="DiT Architecture" style="width: 400px; height: auto;">
</div>

本篇notebook基于如下链接的内容和代码进行整理，参考链接：

[参考知乎解析](https://zhuanlan.zhihu.com/p/684125968)

[DiT论文](https://arxiv.org/pdf/2212.09748)

[DiT模型源码实现](https://github.com/facebookresearch/DiT/blob/main/models.py)

## Timestep embedding

TimestepEmbedder 由一个带有 SiLU 激活和两个线性层的 MLP 构成，嵌入维度默认 256


这里重点解释一下SPE(正弦位置嵌入)，原始公式如下:
$$
PE(pos, 2i) = \sin\left(\frac{pos}{10000^{2i/d_{\text{model}}}}\right)
\\
PE(pos, 2i+1) = \cos\left(\frac{pos}{10000^{2i/d_{\text{model}}}}\right)
$$

但是在实际实现中，采用了如下的公式替换，其中代码的log取负就变成了倒数，那么就能避免除法，直接相乘就可以了:
$$
10000^{\frac{2i}{d_{\text{model}}}} = \left(e^{\log(10000)}\right)^{\frac{2i}{d_{\text{model}}}} = e^{\log(10000) \cdot \frac{2i}{d_{\text{model}}}} = e^{\log(10000) \cdot \frac{i}{\text{half\_d}}}
$$

In [1]:
import torch
import torch.nn as nn
import numpy as np
import math
from timm.models.vision_transformer import PatchEmbed, Attention, Mlp


class TimestepEmbedder(nn.Module):
    """
    Embeds scalar timesteps into vector representations.
    """
    def __init__(self, hidden_size, frequency_embedding_size=256):
        super().__init__()

        self.mlp = nn.Sequential(
            nn.Linear(frequency_embedding_size, hidden_size, bias=True),
            nn.SiLU(),
            nn.Linear(hidden_size, hidden_size, bias=True),
        )
        self.frequency_embedding_size = frequency_embedding_size

    @staticmethod
    def timestep_embedding(t, dim, max_period=10000):
        """
        Create sinusoidal timestep embeddings.

        :param t: a 1-D Tensor of N indices, one per batch element.
                          These may be fractional.
        :param dim: the dimension of the output.
        :param max_period: controls the minimum frequency of the embeddings.
        :return: an (N, D) Tensor of positional embeddings.
        """
        half = dim // 2
        freqs = torch.exp(
            -math.log(max_period) * torch.arange(start=0, end=half, dtype=torch.float32) / half
        ).to(device=t.device)

        # 这里进行了广播操作
        # t: [bs] -> [bs, 1] freqs: [half] -> [1, half]
        args = t[:, None].float() * freqs[None]  
        # 这里和原始的sin/cos交叉存储不一致，采用了sinsin...coscos的结构
        embedding = torch.cat([torch.cos(args), torch.sin(args)], dim=-1)
        
        # dim不是偶数的话就拓展一个0向量
        if dim % 2:
            embedding = torch.cat([embedding, torch.zeros_like(embedding[:, :1])], dim=-1)

        print(f'SPE embedding size: {embedding.size()}')  # [bs, frequency_embedding_size]
        
        return embedding
    
    def forward(self, t):
        # [bs] -> [bs, frequency_embedding_size]
        t_freq = self.timestep_embedding(t, self.frequency_embedding_size)
        # [bs, frequency_embedding_size] -> [bs, hidden_size]
        t_emb = self.mlp(t_freq)
        
        return t_emb

In [2]:
# Timestep Embedding示例
t = torch.tensor([2, 3])
print(f'Before timestep embedding: {t.size()}')

TSE = TimestepEmbedder(768)
t_embedding = TSE(t)
print(f'After timestep embedding: {t_embedding.size()}')

Before timestep embedding: torch.Size([2])
SPE embedding size: torch.Size([2, 256])
After timestep embedding: torch.Size([2, 768])


## Class label embedding

类别的 embedding 由 LabelEmbedder 完成，对 label 输入先通过一定概率概率丢弃来实现 CFG 中的无条件$\emptyset$生成，再由内部 Embedding 层进行嵌入

In [3]:
class LabelEmbedder(nn.Module):
    """
    Embeds class labels into vector representations. Also handles label dropout for classifier-free guidance.
    """
    def __init__(self, num_classes, hidden_size, dropout_prob):
        super().__init__()

        use_cfg_embedding = dropout_prob > 0  # 启用概率丢弃时增加一个空类
        self.embedding_table = nn.Embedding(num_classes + use_cfg_embedding, hidden_size)
        self.num_classes = num_classes
        self.dropout_prob = dropout_prob

    def token_drop(self, labels, force_drop_ids=None):
        """
        Drops labels to enable classifier-free guidance.
        """
        if force_drop_ids is None:
            drop_ids = torch.rand(labels.shape[0], device=labels.device) < self.dropout_prob
        else:
            drop_ids = force_drop_ids == 1

        print(f'Before dropout: {labels}')
        # True的位置表示丢弃，换为空类，False则保留原始类别
        labels = torch.where(drop_ids, self.num_classes, labels)
        print(f'After dropout: {labels}')
        
        return labels
    
    def forward(self, labels, train, force_drop_ids=None):
        use_dropout = self.dropout_prob > 0
        if (train and use_dropout) or (force_drop_ids is not None):
            labels = self.token_drop(labels, force_drop_ids)

        embeddings = self.embedding_table(labels)
        
        return embeddings


In [4]:
num_classes = 1000
dropout_prob = 0.2
CLE = LabelEmbedder(num_classes, 768, dropout_prob)

labels = torch.tensor([114,514, 387, 974, 88, 979])
print(f'Before class label embedding: {labels.size()}')

labels_embedding = CLE(labels, train=True)
print(f'After class label embedding: {labels_embedding.size()}')


Before class label embedding: torch.Size([6])
Before dropout: tensor([114, 514, 387, 974,  88, 979])
After dropout: tensor([ 114,  514,  387,  974, 1000, 1000])
After class label embedding: torch.Size([6, 768])


## DiTBlock
即进行注意力和MLP两层作用，输入和输出维度保持一致

### adaln-zero
在layernorm中，不进行仿射变换，即不学习$\gamma$和$\beta$，因为DiT中使用adaln-zero代替transformer中的normalization，即adaLN_modulation中，输出2组*3个系数，对应$\gamma、\beta、\alpha$，即:
$$
modulate(x, \gamma, \beta) = norm(x) \odot (1 + \gamma) + \beta
\\
output1(x, \alpha 1) = x + attn(modulate(x, \gamma 1, \beta 1))*\alpha 1
\\
output2(output1, \alpha 2) = x + mlp(modulate(output1, \gamma 2, \beta 2))*\alpha 2
$$

### GeLU

timm中的MLP使用tanh来近似GeLU算法，如下公式：
$$
\operatorname{GELU}(x) = 0.5 \cdot x \cdot \left(1 + \tanh\left(\sqrt{\frac{2}{\pi}} \cdot \left(x + 0.044715 \cdot x^3\right)\right)\right)
$$

In [5]:
def modulate(x, shift, scale):
    # 传入的参数shift和scale的维度为[bs, hidden_size]，运算时进行了广播
    # [bs, hidden_size] -> [bs, seq_len, hidden_size]
    return x * (1 + scale.unsqueeze(1)) + shift.unsqueeze(1)

class DiTBlock(nn.Module):
    """
    A DiT block with adaptive layer norm zero (adaLN-Zero) conditioning.
    """
    def __init__(self, hidden_size, num_heads, mlp_ratio=4.0, **block_kwargs):
        super().__init__()

        # elementwise_affine=False，即只进行归一化
        self.norm1 = nn.LayerNorm(hidden_size, elementwise_affine=False, eps=1e-6)
        self.attn = Attention(hidden_size, num_heads=num_heads, qkv_bias=True, **block_kwargs)
        self.norm2 = nn.LayerNorm(hidden_size, elementwise_affine=False, eps=1e-6)
        mlp_hidden_dim = int(hidden_size * mlp_ratio)
        approx_gelu = lambda: nn.GELU(approximate="tanh")
        self.mlp = Mlp(in_features=hidden_size, hidden_features=mlp_hidden_dim, act_layer=approx_gelu, drop=0)
        self.adaLN_modulation = nn.Sequential(
            nn.SiLU(),
            nn.Linear(hidden_size, 6 * hidden_size, bias=True)
        )

    def forward(self, x, c):
        # adaLN_modulation作用之后，在维度通道分出6块[bs, hidden_size]，即对应6组参数
        shift_msa, scale_msa, gate_msa, shift_mlp, scale_mlp, gate_mlp = self.adaLN_modulation(c).chunk(6, dim=1)
        print(f'A param size(e.g. shift_msa): {shift_mlp.size()}')

        # gate参数也进行了广播: [bs, hidden_size] -> [bs, seq_len, hidden_size]
        x = x + gate_msa.unsqueeze(1) * self.attn(modulate(self.norm1(x), shift_msa, scale_msa))
        x = x + gate_mlp.unsqueeze(1) * self.mlp(modulate(self.norm2(x), shift_mlp, scale_mlp))

        return x


In [6]:
batch_size = 4
hidden_size = 768
num_heads = 12
seq_len = 196
DiTBlk = DiTBlock(hidden_size, num_heads)

x = torch.randn(batch_size, seq_len, hidden_size)
t = torch.rand(batch_size)
y = torch.randint(0, 1000, (batch_size,))
t_embedding = TSE(t)  # [bs, hidden_size]
y_embedding = CLE(y, True)  # [bs, hidden_size]
c = t_embedding + y_embedding
print(f'Before DiT Block x size: {x.size()}')
print(f'Before DiT Block c size: {c.size()}')

output = DiTBlk(x, c)
print(f'After DiT Block output size: {output.size()}')



SPE embedding size: torch.Size([4, 256])
Before dropout: tensor([227, 882, 736, 953])
After dropout: tensor([227, 882, 736, 953])
Before DiT Block x size: torch.Size([4, 196, 768])
Before DiT Block c size: torch.Size([4, 768])
A param size(e.g. shift_msa): torch.Size([4, 768])
After DiT Block output size: torch.Size([4, 196, 768])


## Final Layer
FinalLayer在DiT的最后输出latent space表示的$\epsilon_\theta$和$\Sigma_\theta$，其中的normalization也是用了adaln-zero，但是没有学习$\alpha$

最后线性层作用之后，输出维度为patch_size * patch_size * out_channels，即与输入维度z一致

In [7]:
class FinalLayer(nn.Module):
    """
    The final layer of DiT.
    """
    def __init__(self, hidden_size, patch_size, out_channels):
        super().__init__()

        self.norm_final = nn.LayerNorm(hidden_size, elementwise_affine=False, eps=1e-6)
        self.linear = nn.Linear(hidden_size, patch_size * patch_size * out_channels, bias=True)
        self.adaLN_modulation = nn.Sequential(
            nn.SiLU(),
            nn.Linear(hidden_size, 2 * hidden_size, bias=True)
        )

    def forward(self, x, c):
        shift, scale = self.adaLN_modulation(c).chunk(2, dim=1)
        x = modulate(self.norm_final(x), shift, scale)
        x = self.linear(x)
        
        return x
    

In [8]:
patch_size = 4
out_channels = 4
FL = FinalLayer(hidden_size, patch_size, out_channels)

print(f'Before Final Layer size: {output.size()}')
output_final = FL(output, c)
print(f'After Final Layer size: {output_final.size()}')


Before Final Layer size: torch.Size([4, 196, 768])
After Final Layer size: torch.Size([4, 196, 64])


## Patchify
DiT使用timm库的PatchEmbed类将latent z进行patchify来生成符合Transformer 要求的patch token序列，内部通过卷积层实现patchify，举例如下：

假设输入z的维度为(16, 4, 64, 64)，D为1152，patch_size为2，那么卷积层输出:

$$
(\text{input\_size} - \text{kernel\_size})/\text{stride} + 1 = 
\\
(64-2)/2 + 1 = 32
$$
(因为卷积核的尺寸和步长都保持和patch_size一致)

即32*32个patch，输出维度为(16,1152,32,32)

在forward函数中，把其展平为patch序列，还是上面的例子，首先在第三个维度中展平，之后调换2、3维度，得到返回的序列:

(16,1152,32,32) -> (16,1152,32\*32) -> (16,32\*32,32,1152)


In [9]:
pachify_eg_config = {
    'batch_size': 16,
    'in_channels': 4,       
    'hidden_size': 1152,     
    'patch_size': 2,      
    'image_size': 64,       
}

PatchE = PatchEmbed(
    img_size=pachify_eg_config['image_size'],
    patch_size=pachify_eg_config['patch_size'],
    in_chans=pachify_eg_config['in_channels'],
    embed_dim=pachify_eg_config['hidden_size']
)
z = torch.randn(
    pachify_eg_config['batch_size'], 
    pachify_eg_config['in_channels'], 
    pachify_eg_config['image_size'], 
    pachify_eg_config['image_size']
)
print(f'Input z size: {z.size()}')

x = PatchE(z)
print(f'After patchify size: {x.size()}')


Input z size: torch.Size([16, 4, 64, 64])
After patchify size: torch.Size([16, 1024, 1152])


## Patch SPE
Patch的位置编码也使用SPE，通过get_2d_sincos_pos_embed函数实现,根据patch的2d坐标进行位置编码，即行列单独计算正弦位置编码，再进行行列拼接，得到形状为(1, patch_size*patch_size, hidden_size)的SPE，最后再累加到PatchEmbed的patch token


In [10]:
def get_1d_sincos_pos_embed_from_grid(embed_dim, pos):
    """
    embed_dim: output dimension for each position
    pos: a list of positions to be encoded: size (M,)
    
    out: (M, D)
    """
    assert embed_dim % 2 == 0

    omega = np.arange(embed_dim // 2, dtype=np.float64)
    omega /= embed_dim / 2.  # 归一化
    omega = 1. / 10000**omega  # (D/2,), 1/10000^(i/(d/2))

    # 展平为(M,)，即[grid_size, grid_size] -> [grid_size*grid_size]
    pos = pos.reshape(-1)  
    # pos (M,) omega (D/2,), out[m, d] = pos[m] * omega[d]，即外积
    # 或者理解为: pos -> (M, 1) omega -> (1, D/2)，再进行矩阵乘法
    out = np.einsum('m,d->md', pos, omega)  # (M, D/2), outer product

    emb_sin = np.sin(out) # (M, D/2)
    emb_cos = np.cos(out) # (M, D/2)

    emb = np.concatenate([emb_sin, emb_cos], axis=1)  # (M, D)
    
    return emb

def get_2d_sincos_pos_embed_from_grid(embed_dim, grid):
    assert embed_dim % 2 == 0

    # use half of dimensions to encode grid_h
    # grid: [2, 1, grid_size, grid_size]
    # grid[0]/[1]: [1, grid_size, grid_size]
    emb_h = get_1d_sincos_pos_embed_from_grid(embed_dim // 2, grid[0])  # (H*W, D/2)
    emb_w = get_1d_sincos_pos_embed_from_grid(embed_dim // 2, grid[1])  # (H*W, D/2)

    emb = np.concatenate([emb_h, emb_w], axis=1) # (H*W, D)
    
    return emb

def get_2d_sincos_pos_embed(embed_dim, grid_size, cls_token=False, extra_tokens=0):
    """
    grid_size: int of the grid height and width
    
    return:
    pos_embed: [grid_size\*grid_size, embed_dim] or [1+grid_size*grid_size, embed_dim] (w/ or w/o cls_token)
    """
    grid_h = np.arange(grid_size, dtype=np.float32)  # 行坐标(y)↓，[grid_size]
    grid_w = np.arange(grid_size, dtype=np.float32)  # 列坐标(x)→，[grid_size]
    
    # 返回两个[grid_size, grid_size]的array
    # 即grid_w按行复制，grid_h转置为[1, grid_size]之后按列复制
    # 注意向右为x正方向，向下为y正方向
    grid = np.meshgrid(grid_w, grid_h)  # here w goes first
    
    # [2, grid_size, grid_size]
    # dim0为所有对应位置x坐标，dim1为所有对应位置y坐标
    grid = np.stack(grid, axis=0)

    # 这里在dim1添加一个batch维度
    grid = grid.reshape([2, 1, grid_size, grid_size])
    # [2, 1, grid_size, grid_size] -> [grid_size*grid_size, embed_dim]
    pos_embed = get_2d_sincos_pos_embed_from_grid(embed_dim, grid)
    if cls_token and extra_tokens > 0:
        # cls token是可学习的分类标记，拼接在patch序列的开头
        pos_embed = np.concatenate([np.zeros([extra_tokens, embed_dim]), pos_embed], axis=0)
    
    return pos_embed


In [11]:
embed_dim = 768
grid_size = 32

# -> [grid_size^2, embed_dim]
pos_embed = get_2d_sincos_pos_embed(embed_dim, grid_size)
print(f'patch SPE size: {pos_embed.shape}')

patch SPE size: (1024, 768)


## DiT完整模块
到这里一切都清晰了，把上面的所有组件和工具函数进行整合，就得到了完整的DiT模型

### adaLN-Zero调制层初始化
在训练初始阶段，adaLN_modulation的线性层会初始化为0，变为恒等映射(**input_x -> input_x，输入输出一致**)，即$\gamma、\beta、\alpha$都为0，让训练更稳定

### Unpatchify
即patchify的逆操作，把patch序列变回2D的图像格式([batch_size, channels, H, W])

### forwad with cfg(这里不作为重点阐述)
仅用于采样，先调用forward生成out，再把out分成eps和rest两部分，eps作为$\epsilon_\theta$，rest作为后续待处理的方差项$\Sigma_\theta$，依据论文的描述，只对$\epsilon_\theta$进行CFG


In [12]:
class DiT(nn.Module):
    """
    Diffusion model with a Transformer backbone.
    """
    def __init__(
        self,
        input_size=32,
        patch_size=2,
        in_channels=4,
        hidden_size=1152,
        depth=28,
        num_heads=16,
        mlp_ratio=4.0,
        class_dropout_prob=0.1,
        num_classes=1000,
        learn_sigma=True,
    ):
        super().__init__()

        self.learn_sigma = learn_sigma
        self.in_channels = in_channels
        self.out_channels = in_channels * 2 if learn_sigma else in_channels
        self.patch_size = patch_size
        self.num_heads = num_heads

        self.x_embedder = PatchEmbed(input_size, patch_size, in_channels, hidden_size, bias=True)
        self.t_embedder = TimestepEmbedder(hidden_size)
        self.y_embedder = LabelEmbedder(num_classes, hidden_size, class_dropout_prob)
        num_patches = self.x_embedder.num_patches
        # Will use fixed sin-cos embedding:
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches, hidden_size), requires_grad=False)

        self.blocks = nn.ModuleList([
            DiTBlock(hidden_size, num_heads, mlp_ratio=mlp_ratio) for _ in range(depth)
        ])
        self.final_layer = FinalLayer(hidden_size, patch_size, self.out_channels)
        
        self.initialize_weights()

    def initialize_weights(self):
        # Initialize transformer layers:
        # 对所有nn.Linear层进行Xavier均匀初始化 -> 保持输入输出方差一致
        # 偏置初始化为0
        def _basic_init(module):
            if isinstance(module, nn.Linear):
                torch.nn.init.xavier_uniform_(module.weight)
                if module.bias is not None:
                    nn.init.constant_(module.bias, 0)
        self.apply(_basic_init)  # 递归对所有子模块应用_basic_init函数

        # Initialize (and freeze) pos_embed by sin-cos embedding:
        # 正余弦位置编码，不参与训练
        pos_embed = get_2d_sincos_pos_embed(self.pos_embed.shape[-1], int(self.x_embedder.num_patches ** 0.5))
        self.pos_embed.data.copy_(torch.from_numpy(pos_embed).float().unsqueeze(0))

        # Initialize patch_embed like nn.Linear (instead of nn.Conv2d):
        # 展平权重为(out_channels, in_channels * k_h * k_w)后
        # 用Xavier初始化，偏置则为0
        w = self.x_embedder.proj.weight.data
        nn.init.xavier_uniform_(w.view([w.shape[0], -1]))
        nn.init.constant_(self.x_embedder.proj.bias, 0)

        # Initialize label embedding table:
        # 标准差0.02的正态分布初始化
        nn.init.normal_(self.y_embedder.embedding_table.weight, std=0.02)

        # Initialize timestep embedding MLP:
        # 两层线性层使用标准差0.02的正态分布初始化
        nn.init.normal_(self.t_embedder.mlp[0].weight, std=0.02)
        nn.init.normal_(self.t_embedder.mlp[2].weight, std=0.02)

        # Zero-out adaLN modulation layers in DiT blocks:
        # NOTE: DiT Block adaLN调制层的最后一层(线性层)初始化为0，使得其初始为恒等映射，训练更稳定
        for block in self.blocks:
            nn.init.constant_(block.adaLN_modulation[-1].weight, 0)
            nn.init.constant_(block.adaLN_modulation[-1].bias, 0)

        # Zero-out output layers:
        # NOTE: 输出层的adaLN调制层和输出线性层也都初始化为0
        nn.init.constant_(self.final_layer.adaLN_modulation[-1].weight, 0)
        nn.init.constant_(self.final_layer.adaLN_modulation[-1].bias, 0)
        nn.init.constant_(self.final_layer.linear.weight, 0)
        nn.init.constant_(self.final_layer.linear.bias, 0)

    def unpatchify(self, x):
        """
        x: (N, T, patch_size**2 * C)

        imgs: (N, H, W, C)
        """
        c = self.out_channels
        p = self.x_embedder.patch_size[0]
        h = w = int(x.shape[1] ** 0.5)
        assert h * w == x.shape[1]

        x = x.reshape(shape=(x.shape[0], h, w, p, p, c))
        x = torch.einsum('nhwpqc->nchpwq', x)
        imgs = x.reshape(shape=(x.shape[0], c, h * p, h * p))
        
        return imgs
        
    def forward(self, x, t, y):
        """
        Forward pass of DiT.
        x: (N, C, H, W) tensor of spatial inputs (images or latent representations of images)
        t: (N,) tensor of diffusion timesteps
        y: (N,) tensor of class labels
        """
        # pachify
        x = self.x_embedder(x) + self.pos_embed  # (N, T, D), where T = H * W / patch_size ** 2
        
        t = self.t_embedder(t)                   # (N, D)
        y = self.y_embedder(y, self.training)    # (N, D)
        c = t + y                                # (N, D)
        
        for block in self.blocks:
            x = block(x, c)                      # (N, T, D)
        x = self.final_layer(x, c)                # (N, T, patch_size ** 2 * out_channels)
        
        x = self.unpatchify(x)                   # (N, out_channels, H, W)
        
        return x
    
    def forward_with_cfg(self, x, t, y, cfg_scale):
        """
        Forward pass of DiT, but also batches the unconditional forward pass for classifier-free guidance.
        """
        # https://github.com/openai/glide-text2im/blob/main/notebooks/text2im.ipynb
        half = x[: len(x) // 2]
        combined = torch.cat([half, half], dim=0)
        model_out = self.forward(combined, t, y)
        # For exact reproducibility reasons, we apply classifier-free guidance on only
        # three channels by default. The standard approach to cfg applies it to all channels.
        # This can be done by uncommenting the following line and commenting-out the line following that.
        # eps, rest = model_out[:, :self.in_channels], model_out[:, self.in_channels:]
        eps, rest = model_out[:, :3], model_out[:, 3:]
        cond_eps, uncond_eps = torch.split(eps, len(eps) // 2, dim=0)
        half_eps = uncond_eps + cfg_scale * (cond_eps - uncond_eps)
        eps = torch.cat([half_eps, half_eps], dim=0)

        return torch.cat([eps, rest], dim=1)


In [13]:
# 这里使用DiT_XL_2来进行演示
def DiT_XL_2(**kwargs):
    return DiT(depth=28, hidden_size=1152, patch_size=2, num_heads=16, **kwargs)


DiT_model = DiT_XL_2()
batch_size = 4
latent_size = 32
in_channels = 4
x = torch.randn(batch_size, in_channels, latent_size, latent_size)
t = torch.randint(0, 1000, (batch_size,))
y = torch.tensor([207, 360, 387, 974])
print(f'x size: {x.size()}')
print(f't size: {t.size()}')
print(f'y size: {y.size()}')


DiT_model.eval()
output = DiT_model(x, t, y)
print(f'After forward: {output.size()}')


x size: torch.Size([4, 4, 32, 32])
t size: torch.Size([4])
y size: torch.Size([4])
SPE embedding size: torch.Size([4, 256])
A param size(e.g. shift_msa): torch.Size([4, 1152])
A param size(e.g. shift_msa): torch.Size([4, 1152])
A param size(e.g. shift_msa): torch.Size([4, 1152])
A param size(e.g. shift_msa): torch.Size([4, 1152])
A param size(e.g. shift_msa): torch.Size([4, 1152])
A param size(e.g. shift_msa): torch.Size([4, 1152])
A param size(e.g. shift_msa): torch.Size([4, 1152])
A param size(e.g. shift_msa): torch.Size([4, 1152])
A param size(e.g. shift_msa): torch.Size([4, 1152])
A param size(e.g. shift_msa): torch.Size([4, 1152])
A param size(e.g. shift_msa): torch.Size([4, 1152])
A param size(e.g. shift_msa): torch.Size([4, 1152])
A param size(e.g. shift_msa): torch.Size([4, 1152])
A param size(e.g. shift_msa): torch.Size([4, 1152])
A param size(e.g. shift_msa): torch.Size([4, 1152])
A param size(e.g. shift_msa): torch.Size([4, 1152])
A param size(e.g. shift_msa): torch.Size([4,